# Vehicle Data Forensic Cleaning — Fixed Pipeline

**All fixes over the original `clean_vehicle_data_FINAL_v3_FIXED.py`:**
- 🐛 **Bug 1 fixed:** `_flag_imputed` column was completely missing — added
- 🐛 **Bug 2 fixed:** Missing masks now snapshotted *before* imputation so the flag is accurate
- 🐛 **Bug 3 fixed:** `ConditionValue` was missing from `IMPUTABLE_COLS` — rows with missing ConditionValue were invisible to the flag
- 🐛 **Bug 4 fixed:** Sentinel check only caught `'---'` (3 ASCII dashes); data also contains `'—'` (U+2014 em-dash) — now catches all dash variants
- ✅ Sanity check added: no row should be both imputed and non-imputable

**Output flags:**
| Flag | Meaning |
|---|---|
| `_flag_imputed = True` | ≥1 field was filled in by the pipeline |
| `_flag_non_imputable = True` | Make/Model/Trim/Body all NULL, no VIN — nothing could be done |
| Both `False` | Row was complete to begin with |


## 1 · Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
import time

warnings.filterwarnings('ignore')
np.random.seed(42)

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *args): print(f'  ✓ {self.label}: {time.time()-self.t:.2f}s')

audit_log = []
def log(msg):
    audit_log.append(msg)
    print(f'  [{msg[:6]}] {msg}')

print('Imports OK')

Imports OK


## 2 · Configuration & Load

In [2]:
INPUT_FILE  = '/home/mennatullah/Documents/repos/li/AI/instant/hackathon/data/raw/VehicleSales.parquet'   # ← change if needed
OUTPUT_FILE = '/home/mennatullah/Documents/repos/li/AI/instant/hackathon/data/cleaned/v2/VehicleSales_CLEANED.parquet'

with Timer('Load parquet'):
    df = pd.read_parquet(INPUT_FILE)
    print(f'  Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
    null_before = df.isnull().sum()

df.head(3)

  Loaded 558,837 rows x 17 columns
  ✓ Load parquet: 0.33s


,Id,Year,Make,Model,Trim,Body,Transmission,VIN,State,ConditionValue,Odometer,Color,Interior,Seller,MMR,SellingPrice,SaleDate
0,2,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,2014-12-16 04:30:00
1,3,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,2014-12-16 04:30:00
2,4,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,2015-01-14 20:30:00


## 3 · Snapshot Pre-Imputation Missingness ⚠️

> **Bug fix:** We must capture which cells are `NaN` *right now*, before any filling happens.
> After imputation the NaNs are gone and we can no longer tell what was originally missing.

In [3]:
# Bug fix 3: ConditionValue was excluded from IMPUTABLE_COLS, so missing
# ConditionValue rows were never tracked and _flag_imputed stayed False for them.
IMPUTABLE_COLS = [c for c in
    ['Make', 'Model', 'Trim', 'Body', 'Color', 'Interior',
     'Transmission', 'Odometer', 'ConditionValue']
    if c in df.columns]

# Boolean DataFrame — True where value is currently missing
pre_imputation_missing = df[IMPUTABLE_COLS].isna().copy()

print('Missing counts before any cleaning:')
print(pre_imputation_missing.sum().to_string())

Missing counts before any cleaning:
Make              10301
Model             10399
Trim              10651
Body              13195
Color               749
Interior            749
Transmission      65352
Odometer             94
ConditionValue    11820


## 4 · Phase 1 — Structural Fixes (Column Shifts)

In [4]:
# ── CS-001: "Sedan" in Transmission column ───────────────────────────────
with Timer('CS-001 column shift'):
    shift_mask = df['Transmission'].isin(['sedan', 'Sedan'])
    count = shift_mask.sum()
    if count > 0:
        orig = df.loc[shift_mask, ['Transmission', 'VIN', 'State']].copy()
        df.loc[shift_mask, 'State']          = np.nan
        df.loc[shift_mask, 'VIN']            = orig['State'].values
        df.loc[shift_mask, 'Transmission']   = orig['VIN'].values
        df.loc[shift_mask, 'Body']           = 'Sedan'
        df.loc[shift_mask, 'ConditionValue'] = np.nan
        log(f'CS-001: Fixed {count} column-shifted rows')
    else:
        print('  No column shifts detected')

  [CS-001] CS-001: Fixed 26 column-shifted rows
  ✓ CS-001 column shift: 1.59s


In [5]:
# ── CS-002: Make-body concatenations ─────────────────────────────────────
with Timer('CS-002 make-body concat'):
    make_body_map = {
        'ford truck': ('Ford', 'Truck'),   'gmc truck':  ('GMC',       'Truck'),
        'chev truck': ('Chevrolet','Truck'),'chev tk':   ('Chevrolet', 'Truck'),
        'ford tk':    ('Ford',    'Truck'), 'dodge tk':  ('Dodge',     'Truck'),
        'mazda tk':   ('Mazda',   'Truck'), 'hyundai tk':('Hyundai',   'Truck'),
    }
    make_lower = df['Make'].str.lower()
    for raw, (mk, bd) in make_body_map.items():
        mask = make_lower == raw
        if mask.sum() > 0:
            df.loc[mask, 'Make'] = mk
            df.loc[mask, 'Body'] = bd
            log(f'CS-002: "{raw}" → {mk}/{bd} ({mask.sum()} rows)')

  [CS-002] CS-002: "ford truck" → Ford/Truck (3 rows)
  [CS-002] CS-002: "gmc truck" → GMC/Truck (11 rows)
  [CS-002] CS-002: "chev truck" → Chevrolet/Truck (1 rows)
  [CS-002] CS-002: "ford tk" → Ford/Truck (1 rows)
  [CS-002] CS-002: "dodge tk" → Dodge/Truck (1 rows)
  [CS-002] CS-002: "mazda tk" → Mazda/Truck (1 rows)
  [CS-002] CS-002: "hyundai tk" → Hyundai/Truck (1 rows)
  ✓ CS-002 make-body concat: 5.55s


In [6]:
# ── MF-001: Range Rover fragmentation ────────────────────────────────────
with Timer('MF-001 Range Rover'):
    model_lower = df['Model'].str.lower()
    m1 = (model_lower == 'range') & df['Trim'].str.contains('rover', case=False, na=False)
    m2 = model_lower == 'rangerover'
    m3 = df['Model'].isin(['rr', 'RR', 'rrs', 'RRS'])
    rr_mask = m1 | m2 | m3
    if rr_mask.sum() > 0:
        df.loc[rr_mask, 'Model'] = 'Range Rover'
        log(f'MF-001: Fixed {rr_mask.sum()} Range Rover rows')

  [MF-001] MF-001: Fixed 87 Range Rover rows
  ✓ MF-001 Range Rover: 0.53s


## 5 · Phase 2 — Standardization

In [7]:
with Timer('MK-001 make standardization'):
    make_mapping = {
        'vw': 'Volkswagen', 'dot': 'Dodge',
        'mercedes-b': 'Mercedes-Benz', 'mercedes-benz': 'Mercedes-Benz',
        'chev': 'Chevrolet', 'chevy': 'Chevrolet',
    }
    df['Make'] = df['Make'].str.lower().replace(make_mapping).str.title()
    log('MK-001: Make names standardized')

with Timer('Categorical standardization'):
    trans_map = {'auto': 'automatic', 'man': 'manual'}
    df['Transmission'] = df['Transmission'].str.strip().str.lower().replace(trans_map)
    for col in ['Make', 'Model', 'Body', 'Color', 'Interior', 'State', 'Trim']:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()
    log('STD: Categoricals standardized')

  [MK-001] MK-001: Make names standardized
  ✓ MK-001 make standardization: 0.19s
  [STD: C] STD: Categoricals standardized
  ✓ Categorical standardization: 0.26s


## 6 · Phase 3 — MNAR Sentinel Values → NaN

In [8]:
with Timer('Sentinel conversion'):
    m = df['Odometer'] == 999999
    if m.sum() > 0:
        df.loc[m, 'Odometer'] = np.nan
        log(f'PH-001: {m.sum()} odometer sentinels → NaN')

    # Fixed for all dash variants including em-dash
    SENTINELS = {'---', '—', '–', '———', '\u2014', '\u2013'}
    for col in ['Color', 'Interior']:
        if col in df.columns:
            m = df[col].isin(SENTINELS) | df[col].astype(str).str.strip().isin(SENTINELS)
            if m.sum() > 0:
                df.loc[m, col] = np.nan
                log(f'PH-002: {m.sum()} {col} placeholders → NaN')

    if 'MMR' in df.columns:
        m = df['MMR'] == 999999
        if m.sum() > 0:
            df.loc[m, 'MMR'] = np.nan
            log(f'PH-003: {m.sum()} MMR sentinels → NaN')

  [PH-001] PH-001: 72 odometer sentinels → NaN
  [PH-002] PH-002: 24685 Color placeholders → NaN
  [PH-002] PH-002: 17077 Interior placeholders → NaN
  ✓ Sentinel conversion: 0.79s


## 7 · Phase 4 — VIN-Based Imputation (Domain Knowledge)

In [9]:
WMI_DB = {
    '1FT':('Ford','Truck'),       '1G1':('Chevrolet',None),   '1GC':('Chevrolet','Truck'),
    '1GT':('GMC','Truck'),        '1HG':('Honda',None),        '1J4':('Jeep',None),
    '1J8':('Jeep',None),          '1N4':('Nissan',None),       '1N6':('Nissan','Truck'),
    '1VW':('Volkswagen',None),    '1YV':('Mazda',None),        '19U':('Acura',None),
    '2G1':('Chevrolet',None),     '2T1':('Toyota',None),       '2T3':('Toyota','Suv'),
    '3VW':('Volkswagen',None),    '4S3':('Subaru',None),       '4S4':('Subaru',None),
    '5J6':('Honda','Suv'),        '5N1':('Hyundai','Suv'),     '5UX':('BMW',None),
    '5XX':('Kia',None),           'JM1':('Mazda',None),        'JTD':('Toyota',None),
    'JTH':('Lexus',None),         'JTJ':('Lexus','Suv'),       'KNA':('Kia',None),
    'KNB':('Kia',None),           'KND':('Kia',None),          'KMH':('Hyundai',None),
    'SAL':('Land Rover',None),    'SAJ':('Jaguar',None),       'SCA':('Rolls Royce',None),
    'SCB':('Bentley',None),       'SCF':('Aston Martin',None), 'WA1':('Audi','Suv'),
    'WAU':('Audi',None),          'WBA':('BMW',None),          'WBS':('BMW',None),
    'WBX':('BMW','Suv'),          'WDC':('Mercedes-Benz','Suv'),'WDD':('Mercedes-Benz',None),
    'WP0':('Porsche',None),       'WP1':('Porsche','Suv'),     'WVW':('Volkswagen',None),
    'YV1':('Volvo',None),         'YV4':('Volvo','Suv'),       'ZFF':('Ferrari',None),
    '1FA':('Ford',None),          '1FM':('Ford','Suv'),        '1LN':('Lincoln',None),
    '1B4':('Dodge',None),         '1C3':('Chrysler',None),     '1C4':('Chrysler',None),
    '3N1':('Nissan',None),        '4T1':('Toyota',None),       '2HG':('Honda',None),
    '2HK':('Honda','Suv'),        'JN1':('Nissan',None),       'JF1':('Subaru',None),
    'JF2':('Subaru',None),        'JH4':('Acura',None),        '1GK':('GMC','Suv'),
    'ZAM':('Maserati',None),      'ZAR':('Alfa Romeo',None),   'ZFA':('Fiat',None),
}

with Timer('IMP-VIN vectorized'):
    missing_make = df['Make'].isna() & df['VIN'].notna()
    if missing_make.sum() > 0:
        wmi_series    = df.loc[missing_make, 'VIN'].str[:3].str.upper()
        make_from_wmi = wmi_series.map({k: v[0] for k, v in WMI_DB.items()})
        n_make = make_from_wmi.notna().sum()
        df.loc[missing_make, 'Make'] = df.loc[missing_make, 'Make'].fillna(make_from_wmi)
        missing_both = missing_make & df['Body'].isna()
        if missing_both.sum() > 0:
            wmi_body = df.loc[missing_both, 'VIN'].str[:3].str.upper().map(
                {k: v[1] for k, v in WMI_DB.items()})
            df.loc[missing_both, 'Body'] = df.loc[missing_both, 'Body'].fillna(wmi_body)
        log(f'IMP-VIN: Imputed {n_make} Make values from WMI')

  [IMP-VI] IMP-VIN: Imputed 5323 Make values from WMI
  ✓ IMP-VIN vectorized: 0.82s


## 8 · Phase 5 — Mode Imputation for Color / Interior

In [10]:
with Timer('Mode imputation'):
    for col in ['Color', 'Interior']:
        if col in df.columns:
            missing = df[col].isna()
            if missing.sum() > 0:
                for (make, body), group in df.groupby(['Make', 'Body']):
                    mode_val = group[col].mode()
                    if not mode_val.empty:
                        mask = missing & (df['Make'] == make) & (df['Body'] == body)
                        if mask.sum() > 0:
                            df.loc[mask, col] = mode_val[0]

                still_missing = df[col].isna()
                if still_missing.sum() > 0:
                    overall_mode = df[col].mode()
                    if not overall_mode.empty:
                        df.loc[still_missing, col] = overall_mode[0]

                filled = missing.sum() - df[col].isna().sum()
                log(f'IMP-MODE: {col} - {filled} values imputed by mode')

  [IMP-MO] IMP-MODE: Color - 25434 values imputed by mode
  [IMP-MO] IMP-MODE: Interior - 17826 values imputed by mode
  ✓ Mode imputation: 158.37s


## 9 · Phase 6 — Build Flags

> **Bug fix:** `_flag_imputed` is built by comparing the pre-imputation snapshot
> against the current DataFrame. Any cell that was `NaN` before and has a value now → row is flagged.

In [11]:
with Timer('Build flags'):

    # ── _flag_imputed ──────────────────────────────────────────────────────
    # Was missing before AND is now filled  →  True
    imputed_any = pd.Series(False, index=df.index)
    for col in IMPUTABLE_COLS:
        was_missing = pre_imputation_missing[col]
        now_filled  = df[col].notna()
        imputed_any = imputed_any | (was_missing & now_filled)

    df['_flag_imputed'] = imputed_any
    log(f'FLAG-IMP: {imputed_any.sum():,} rows have ≥1 imputed field')

    # ── _flag_non_imputable ────────────────────────────────────────────────
    # Make/Model/Trim/Body all still NULL with no valid VIN
    cat_cols = [c for c in ['Make', 'Model', 'Trim', 'Body'] if c in df.columns]
    all_null = df[cat_cols].isna().all(axis=1)
    no_vin   = df['VIN'].isna() | (df['VIN'].astype(str).str.strip() == '')
    non_imp  = all_null & no_vin
    df['_flag_non_imputable'] = non_imp
    log(f'FLAG-NONIMP: {non_imp.sum():,} rows flagged as non-imputable')

    # ── Sanity check ───────────────────────────────────────────────────────
    overlap = (imputed_any & non_imp).sum()
    if overlap:
        print(f'  ⚠ WARNING: {overlap} rows flagged as BOTH imputed and non-imputable!')
    else:
        print('  ✓ Sanity check passed: no overlap between the two flags')

  [FLAG-I] FLAG-IMP: 6,084 rows have ≥1 imputed field
  [FLAG-N] FLAG-NONIMP: 0 rows flagged as non-imputable
  ✓ Sanity check passed: no overlap between the two flags
  ✓ Build flags: 0.07s


## 10 · Summary & Null Comparison

In [12]:
null_after = df.isnull().sum()
comparison = pd.DataFrame({
    'Null Before': null_before,
    'Null After':  null_after,
    'Fixed':       null_before - null_after,
})
comparison = comparison[comparison['Null Before'] > 0].sort_values('Fixed', ascending=False)

print('=== NULL COMPARISON ===')
print(comparison.to_string())
print()
print(f'Final shape:        {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Imputed rows:       {df["_flag_imputed"].sum():,}')
print(f'Non-imputable rows: {df["_flag_non_imputable"].sum():,}')
print(f'Clean (unchanged):  {(~df["_flag_imputed"] & ~df["_flag_non_imputable"]).sum():,}')

=== NULL COMPARISON ===
                Null Before  Null After   Fixed
Make                10301.0        4978  5323.0
Body                13195.0       11277  1918.0
Color                 749.0           0   749.0
Interior              749.0           0   749.0
VIN                     4.0           0     4.0
MMR                    38.0          38     0.0
ConditionValue      11820.0       11820     0.0
SaleDate               38.0          38     0.0
Model               10399.0       10399     0.0
Trim                10651.0       10651     0.0
SellingPrice           12.0          12     0.0
Transmission        65352.0       65356    -4.0
Odometer               94.0         166   -72.0

Final shape:        558,837 rows x 19 columns
Imputed rows:       6,084
Non-imputable rows: 0
Clean (unchanged):  552,753


In [13]:
print('=== AUDIT LOG ===')
for entry in audit_log:
    print(f'  • {entry}')

=== AUDIT LOG ===
  • CS-001: Fixed 26 column-shifted rows
  • CS-002: "ford truck" → Ford/Truck (3 rows)
  • CS-002: "gmc truck" → GMC/Truck (11 rows)
  • CS-002: "chev truck" → Chevrolet/Truck (1 rows)
  • CS-002: "ford tk" → Ford/Truck (1 rows)
  • CS-002: "dodge tk" → Dodge/Truck (1 rows)
  • CS-002: "mazda tk" → Mazda/Truck (1 rows)
  • CS-002: "hyundai tk" → Hyundai/Truck (1 rows)
  • MF-001: Fixed 87 Range Rover rows
  • MK-001: Make names standardized
  • STD: Categoricals standardized
  • PH-001: 72 odometer sentinels → NaN
  • PH-002: 24685 Color placeholders → NaN
  • PH-002: 17077 Interior placeholders → NaN
  • IMP-VIN: Imputed 5323 Make values from WMI
  • IMP-MODE: Color - 25434 values imputed by mode
  • IMP-MODE: Interior - 17826 values imputed by mode
  • FLAG-IMP: 6,084 rows have ≥1 imputed field
  • FLAG-NONIMP: 0 rows flagged as non-imputable


## 11 · Flag Breakdown

In [14]:
flag_summary = pd.DataFrame({
    'Category': [
        'Clean (no changes)',
        'Imputed (≥1 field filled)',
        'Non-imputable (flagged, excluded from cat. analysis)',
    ],
    'Row count': [
        (~df['_flag_imputed'] & ~df['_flag_non_imputable']).sum(),
        df['_flag_imputed'].sum(),
        df['_flag_non_imputable'].sum(),
    ]
})
flag_summary['Share %'] = (flag_summary['Row count'] / len(df) * 100).round(2)
flag_summary

,Category,Row count,Share %
0,Clean (no changes),552753,98.91
1,Imputed (≥1 field filled),6084,1.09
2,"Non-imputable (flagged, excluded from cat. ana...",0,0.00


## 12 · Sample Imputed Rows

In [15]:
df[df['_flag_imputed']].head(10)

,Id,Year,Make,Model,Trim,Body,Transmission,VIN,State,ConditionValue,Odometer,Color,Interior,Seller,MMR,SellingPrice,SaleDate,_flag_imputed,_flag_non_imputable
493,495,2013,Mercedes-Benz,M-Class,Ml350,Suv,NaN,4jgda5jb6da160181,Ca,NaN,32532.0,Black,Black,trade in solutions irvine,36100.0,34500.0,2014-12-16 04:30:00,True,False
742,744,2012,BMW,NaN,NaN,NaN,automatic,wbakb8c51cc964387,Ca,38.0,23208.0,Gray,Black,financial services remarketing (lease),47200.0,46000.0,2015-02-25 20:30:00,True,False
747,749,2012,BMW,NaN,NaN,NaN,automatic,wbakb8c53cc964410,Ca,33.0,19785.0,Beige,Gray,financial services remarketing (lease),49500.0,46000.0,2015-02-11 20:30:00,True,False
766,768,2012,BMW,NaN,NaN,NaN,automatic,wbakb8c54cc964089,Ca,37.0,48424.0,Black,Black,financial services remarketing (lease),42300.0,43000.0,2015-01-14 20:30:00,True,False
798,800,2012,BMW,NaN,NaN,NaN,automatic,wbakb8c59cc448049,Ca,48.0,39825.0,Black,Gray,financial services remarketing (lease),58100.0,58500.0,2015-01-14 20:30:00,True,False
803,805,2012,BMW,NaN,NaN,NaN,automatic,wbakb8c58cc962863,Ca,49.0,35093.0,Blue,Tan,financial services remarketing (lease),45200.0,44500.0,2015-01-28 20:30:00,True,False
842,844,2012,BMW,NaN,NaN,NaN,automatic,wbakb8c50cc964249,Ca,4.0,33155.0,Gray,Gray,financial services remarketing (lease),46100.0,45500.0,2015-01-14 20:30:00,True,False
893,895,2012,BMW,NaN,NaN,NaN,automatic,wbakc8c50cc435719,Ca,24.0,27709.0,Black,Black,bmw of bakersfield,41800.0,33750.0,2015-02-11 20:30:00,True,False
1514,1516,2012,Land Rover,NaN,NaN,NaN,automatic,salmf1e46ca360169,Ca,29.0,60828.0,Black,Black,jpmorgan chase bank n.a.,47600.0,40500.0,2015-01-14 20:30:00,True,False
1539,1541,2012,Land Rover,NaN,NaN,NaN,automatic,salmf1e48ca360609,Ca,29.0,62786.0,Black,Black,jpmorgan chase bank n.a.,46400.0,42500.0,2015-02-11 20:30:00,True,False


## 13 · Save Output

In [17]:
df.to_parquet(OUTPUT_FILE, index=False)
print(f'✓ Saved to {OUTPUT_FILE}')
print(f'  Size in memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

✓ Saved to /home/mennatullah/Documents/repos/li/AI/instant/hackathon/data/cleaned/v2/VehicleSales_CLEANED.parquet
  Size in memory: 117.4 MB


In [18]:
import pandas as pd
df = pd.read_parquet('/home/mennatullah/Documents/repos/li/AI/instant/hackathon/data/cleaned/v2/VehicleSales_CLEANED.parquet')                                            
df.to_csv('vehicle_sales_dashboard.csv', index=False)